In [ ]:
# Section 1 — Environment check (Python, pip, kernel)
import sys
import subprocess
print('Python executable:', sys.executable)
print('Python version:', sys.version)
res = subprocess.run([sys.executable, '-m', 'pip', '--version'], capture_output=True, text=True)
print(res.stdout.strip())
res2 = subprocess.run([sys.executable, '-m', 'pip', 'list'], capture_output=True, text=True)
lines = res2.stdout.splitlines()
keywords = ['numpy','scipy','patsy','statsmodels','pandas']
print('\nInstalled packages (filtered):')
for line in lines:
    low = line.lower()
    if any(k in low for k in keywords):
        print(line)


In [ ]:
# Section 2 — Install statsmodels (notebook-friendly)
# Run these lines in the notebook cell (they use the kernel's pip)
%pip install --upgrade pip setuptools wheel
%pip install statsmodels

# Note: Restart the kernel after successful install: Kernel -> Restart (or Notebook -> Restart Kernel)


# Section 3 — Alternative: conda install (instructions for terminal)

"""
If you use conda, run these in an integrated terminal (adjust env name):

conda activate <env>
conda install -c conda-forge statsmodels patsy

# Or create a new env and install:
conda create -n viz2 python=3.10 statsmodels -c conda-forge && conda activate viz2
"""


In [ ]:
# Section 4 — Verify import and show installed versions
import importlib.util
spec = importlib.util.find_spec('statsmodels')
print('statsmodels found:', spec is not None)
if spec is not None:
    import statsmodels as sm
    import numpy as np, pandas as pd
    print('statsmodels.__version__ =', sm.__version__)
else:
    print('statsmodels not importable in this kernel yet. Restart kernel after installing and run this cell again.')


In [ ]:
# Section 5 — Minimal statsmodels OLS example (synthetic data)
import numpy as np
import pandas as pd
try:
    import statsmodels.api as sm
except Exception:
    print('statsmodels not available; run the install cell and restart kernel')
else:
    n = 100
    X = np.column_stack([np.ones(n), np.random.normal(size=n)])
    y = 1.5 + 2.0*X[:,1] + np.random.normal(scale=0.5, size=n)
    res = sm.OLS(y, X).fit(cov_type='HC1')
    print(res.summary())


In [ ]:
# Section 6 — Add graceful fallback in your script (try/except + flag)
# Run this in a cell or add to top of your vis2 script/notebook
try:
    import statsmodels.api as sm
    STATS_MODELS_AVAILABLE = True
except Exception:
    STATS_MODELS_AVAILABLE = False

print('STATS_MODELS_AVAILABLE =', STATS_MODELS_AVAILABLE)

# Guard model sections in your script like:
# if not STATS_MODELS_AVAILABLE:
#     print('Skipping mechanism models: statsmodels not installed')


In [ ]:
# Section 7 — Smoke test of fit_twoway_fe using synthetic panel
import numpy as np
import pandas as pd
import importlib

try:
    from vis2 import fit_twoway_fe
    can_import_vis2 = True
except Exception as e:
    print('Could not import fit_twoway_fe from vis2:', e)
    can_import_vis2 = False

# Build minimal panel
n_hosp = 5
years = list(range(2018, 2023))
df = pd.DataFrame({
    'IK': np.repeat(np.arange(1, n_hosp+1), len(years)),
    'year': years * n_hosp,
})
np.random.seed(1)
df['nurseper1kpatientday'] = np.random.normal(5,1,size=len(df))
df['mdper1kpatientday'] = np.random.normal(2,0.5,size=len(df))
df['pandemic'] = (df['year'] >= 2020).astype(int)
df['index_all_items'] = 3 + 0.2*df['pandemic'] + np.random.normal(0,0.3,size=len(df))

if can_import_vis2:
    try:
        resA, resB, _ = fit_twoway_fe(df, y='index_all_items', xcols=['pandemic'], entity='IK', time='year', weight_col=None, cluster_col='IK')
        print(resA.summary())
    except Exception as e:
        print('Error running fit_twoway_fe smoke test:', e)
else:
    print('Skipping smoke test: fit_twoway_fe not importable from vis2 (not a module).')


# Section 8 — Persist changes & run checklist
"""
Checklist:
- Save this notebook (File -> Save).
- Run Section 1 to inspect current interpreter and installed packages.
- Run Section 2 to install statsmodels into the kernel (or run the conda commands in Section 3).
- Restart the kernel (Kernel -> Restart) after installs.
- Run Sections 4-7 to verify import and run smoke tests.
- If you want, I can apply the try/except fallback directly into your main vis2 notebook/script; say "apply patch" and I'll update `vis2.ipynb`.
"""
